In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗# ║  Credit Limit Causal Analysis v8 — Implementation Plan 20260324          ║# ║  Part A: PfP existence (DML event + hazard)                               ║# ║  Part B: Excess-inc local causal effect                                   ║# ║  Part C: Score-to-treatment rule analysis                                 ║# ║  All regressions: STATA-format output (0.1:* / 0.05:** / 0.01:***)       ║# ║  All credit variables: standardised before entering models               ║# ╚══════════════════════════════════════════════════════════════════════════╝# ╔══════════════════════════════════════════════════════════════════════════╗# ║  SECTION A: CONFIGURATION & IMPORTS                                       ║# ╚══════════════════════════════════════════════════════════════════════════╝import warnings; warnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy import statsfrom scipy.stats import ks_2sampfrom sklearn.model_selection import GroupKFold, StratifiedKFoldfrom sklearn.metrics import roc_auc_score, roc_curvefrom sklearn.linear_model import LinearRegression, LogisticRegressionfrom sklearn.neighbors import NearestNeighborsimport statsmodels.api as smimport statsmodels.formula.api as smffrom IPython.display import display, HTMLimport xgboost as xgbimport os, itertoolsos.environ["CUDA_VISIBLE_DEVICES"] = "1"plt.rcParams.update({    "font.family": "DejaVu Sans", "font.size": 10, "figure.dpi": 150,    "axes.titlesize": 11, "axes.labelsize": 10,    "figure.figsize": (10, 5), "legend.fontsize": 8,})sns.set_style("whitegrid")# ── Master Configuration ──────────────────────────────────────────────────CFG = dict(    data_path       = "working_data/2wcase_updated.csv",    model_path_v4   = "working_data/model/v4-短期.model",    feature_path_v4 = "working_data/model/v4-短期.xlsx",    model_path_v5   = "working_data/model/v5-短期.model",    feature_path_v5 = "working_data/model/v5-短期.xlsx",    cols = dict(        user_id    = "user_id",        event_date = "first_done_date",        bf_credit  = "first_bf_credit",        af_credit  = "first_af_credit",        model_type = "model_type",        label1     = "mob1_label7",        label2     = "mob2_label7",    ),    overlap_start     = pd.Timestamp("2025-03-20"),    overlap_end       = pd.Timestamp("2025-06-05"),    max_followup_days = 60,    n_folds           = 5,    n_repeats         = 10,    seed              = 42,    xgb_device        = "cuda",    alpha             = 0.05,)C = CFG["cols"]print("Configuration loaded.")print(f"  Overlap window : {CFG['overlap_start'].date()} -> {CFG['overlap_end'].date()}")print(f"  Max follow-up  : {CFG['max_followup_days']} days")# ╔══════════════════════════════════════════════════════════════════════════╗# ║  UTILITY FUNCTIONS                                                        ║# ╚══════════════════════════════════════════════════════════════════════════╝def stars(p):    if p < 0.01: return "***"    if p < 0.05: return "**"    if p < 0.10: return "*"    return ""def stata_table(model, title="", note="", extra_info=None):    """STATA-style regression table: * p<0.10, ** p<0.05, *** p<0.01"""    W = 80    print(f"\n{'─'*W}")    if title:        print(f"  {title}")        print(f"{'─'*W}")    print(f"  {'Variable':.<30s} {'Coef.':>10} {'SE':>10} {'z':>8} {'p':>9}  {'95% CI':>24}")    print(f"{'─'*W}")    ci = model.conf_int()    for v in model.params.index:        c  = model.params[v]; se = model.bse[v]        z  = c / se if se > 0 else 0; p = model.pvalues[v]        lo, hi = ci.loc[v, 0], ci.loc[v, 1]        print(f"  {v:.<30s} {c:>10.4f} {se:>10.4f} {z:>8.2f} {p:>9.4f}  [{lo:>9.4f}, {hi:>9.4f}] {stars(p)}")    print(f"{'─'*W}")    n_str = f"N={int(model.nobs):,}"    aic_str = f"AIC={model.aic:.1f}" if hasattr(model, 'aic') else ""    print(f"  {n_str}  {aic_str}")    if extra_info:        for k, v in extra_info.items():            print(f"  {k}: {v}")    if note:        print(f"  Note: {note}")    print("  * p<0.10  ** p<0.05  *** p<0.01")def dml_table(name, theta, se, z, p, ci_lo, ci_hi, n, extra_info=None):    """STATA-style table for DML point estimates."""    W = 80    print(f"\n{'─'*W}")    print(f"  {name}")    print(f"{'─'*W}")    print(f"  {'Variable':.<30s} {'Coef.':>10} {'SE':>10} {'z':>8} {'p':>9}  {'95% CI':>24}")    print(f"{'─'*W}")    if isinstance(theta, dict):        for var in theta:            c = theta[var]; s = se[var]; zv = z[var]; pv = p[var]            lo = ci_lo[var]; hi = ci_hi[var]            print(f"  {var:.<30s} {c:>10.4f} {s:>10.4f} {zv:>8.2f} {pv:>9.4f}  [{lo:>9.4f}, {hi:>9.4f}] {stars(pv)}")    else:        print(f"  {'treatment':.<30s} {theta:>10.4f} {se:>10.4f} {z:>8.2f} {p:>9.4f}  [{ci_lo:>9.4f}, {ci_hi:>9.4f}] {stars(p)}")    print(f"{'─'*W}")    print(f"  N={n:,}")    if extra_info:        for k, v in extra_info.items():            print(f"  {k}: {v}")    print("  * p<0.10  ** p<0.05  *** p<0.01")def safe_glm(formula, data, family=None, cluster_col=None):    if family is None:        family = sm.families.Binomial()    try:        if cluster_col and cluster_col in data.columns:            m = smf.glm(formula, data=data, family=family).fit(                cov_type="cluster", cov_kwds={"groups": data[cluster_col]})        else:            m = smf.glm(formula, data=data, family=family).fit(cov_type="HC1")        return m    except Exception as e:        print(f"  [GLM failed] {formula[:60]}... - {e}")        return Nonedef kaplan_meier(T, E):    dk = pd.DataFrame({"T": T, "E": E}).dropna()    ts = sorted(dk["T"].unique()); nr = len(dk); s = 1.0    out = [{"time": 0, "survival": 1.0}]    for t in ts:        d = ((dk["T"] == t) & (dk["E"] == 1)).sum()        c_ = ((dk["T"] == t) & (dk["E"] == 0)).sum()        if nr > 0: s *= (1 - d / nr)        out.append({"time": t, "survival": s}); nr -= (d + c_)    return pd.DataFrame(out)def logrank_test(T1, E1, T2, E2):    d1 = pd.DataFrame({"T": T1, "E": E1, "g": 0}).dropna()    d2 = pd.DataFrame({"T": T2, "E": E2, "g": 1}).dropna()    da = pd.concat([d1, d2])    ts = sorted(da.loc[da["E"] == 1, "T"].unique())    O1 = E1e = V = 0.0    for t in ts:        ar = da[da["T"] >= t]; n = len(ar); n1 = (ar["g"] == 0).sum()        d = ((ar["T"] == t) & (ar["E"] == 1)).sum()        d1v = ((ar["T"] == t) & (ar["E"] == 1) & (ar["g"] == 0)).sum()        if n < 2: continue        e1 = n1 * d / n; O1 += d1v; E1e += e1        V += n1 * (n - n1) * d * (n - d) / (n**2 * (n - 1)) if n > 1 else 0    if V <= 0: return np.nan, np.nan    chi2 = (O1 - E1e) ** 2 / V    return chi2, 1 - stats.chi2.cdf(chi2, 1)def compute_smd(t, c):    t = pd.Series(t).dropna(); c = pd.Series(c).dropna()    if len(t) < 2 or len(c) < 2: return np.nan    ps = np.sqrt((t.var() + c.var()) / 2)    return (t.mean() - c.mean()) / ps if ps > 0 else 0def safe_qcut_rank(s, q, labels=False):    r = pd.Series(s).rank(method="first")    q_eff = min(q, r.nunique())    if q_eff < 2:        return pd.Series(np.zeros(len(r), dtype=int), index=r.index)    return pd.qcut(r, q=q_eff, labels=labels, duplicates="drop")def xgb_crossfit(df, feat_list, target, cfg, group_col=None,                 max_depth=4, colsample=0.6, mcw=20, num_rounds=300,                 objective="binary:logistic", is_binary=True):    """Cross-fitted OOF predictions from XGBoost.    XGBoost handles NaN natively; GroupKFold by user_id prevents leakage."""    fl = [f for f in feat_list if f in df.columns]    if not fl:        return np.full(len(df), np.nan), np.nan, np.nan, 0    X = df[fl].values.astype(np.float32)    y = df[target].values.astype(int if is_binary else float)    oof = np.full(len(df), np.nan)    use_group = (group_col is not None and group_col in df.columns                 and df[group_col].nunique() >= cfg["n_folds"])    if use_group:        splitter = GroupKFold(n_splits=cfg["n_folds"])        split_iter = splitter.split(X, y, groups=df[group_col].values)    else:        splitter = StratifiedKFold(n_splits=cfg["n_folds"], shuffle=True,                                   random_state=cfg["seed"])        split_iter = splitter.split(X, y if is_binary else np.zeros(len(y)))    params = dict(        objective=objective, eval_metric="logloss" if is_binary else "rmse",        max_depth=max_depth, learning_rate=0.05, subsample=0.8,        colsample_bytree=colsample, min_child_weight=mcw,        reg_alpha=0.1, reg_lambda=1.0,        device=cfg.get("xgb_device", "cpu"), verbosity=0,    )    aucs = []    for tr_i, va_i in split_iter:        dt = xgb.DMatrix(X[tr_i], label=y[tr_i], feature_names=fl, missing=np.nan)        dv = xgb.DMatrix(X[va_i], label=y[va_i], feature_names=fl, missing=np.nan)        bst = xgb.train(params, dt, num_boost_round=num_rounds,                        evals=[(dv, "v")], early_stopping_rounds=30,                        verbose_eval=False)        oof[va_i] = bst.predict(dv)        if is_binary and len(np.unique(y[va_i])) == 2:            aucs.append(roc_auc_score(y[va_i], oof[va_i]))    mean_m = np.mean(aucs) if aucs else np.nan    std_m = np.std(aucs) if aucs else np.nan    return oof, mean_m, std_m, len(fl)def standardise(s):    """Z-score standardisation."""    mu = s.mean(); sd = s.std()    return (s - mu) / max(sd, 1e-8)print("Utility functions loaded.")# ╔══════════════════════════════════════════════════════════════════════════╗# ║  SECTION B: DATA LOADING & PREPROCESSING                                  ║# ╚══════════════════════════════════════════════════════════════════════════╝raw = pd.read_csv(CFG["data_path"], low_memory=False)print(f"Loaded {len(raw):,} rows, {len(raw.columns)} columns.")def basic_clean(df):    df = df.copy()    df[C["event_date"]] = pd.to_datetime(df[C["event_date"]], errors="coerce")    for col in [C["bf_credit"], C["af_credit"]]:        df[col] = pd.to_numeric(df[col], errors="coerce")    for col in [C["label1"], C["label2"]]:        df[col] = pd.to_numeric(df[col], errors="coerce") if col in df.columns else np.nan    df = df.dropna(subset=[C["event_date"], C["bf_credit"], C["af_credit"]]).copy()    df["event_date"] = df[C["event_date"]].dt.normalize()    df["treat_dir"] = np.select(        [df[C["af_credit"]] > df[C["bf_credit"]],         df[C["af_credit"]] == df[C["bf_credit"]],         df[C["af_credit"]] < df[C["bf_credit"]]],        ["inc", "same", "dec"], default="unknown",    )    df = df[df["treat_dir"].isin(["inc", "same", "dec"])].copy()    df["is_inc"] = (df["treat_dir"] == "inc").astype(int)    df["is_dec"] = (df["treat_dir"] == "dec").astype(int)    df["delta_abs"] = df[C["af_credit"]] - df[C["bf_credit"]]    bf_safe = df[C["bf_credit"]].clip(lower=1.0)    df["delta_rel"] = df["delta_abs"] / bf_safe    df["delta_log"] = np.log(df[C["af_credit"]].clip(lower=1.0) / bf_safe)    if C["model_type"] in df.columns:        mt = df[C["model_type"]].astype(str).str.upper()        df["model_family"] = np.where(mt.str.contains("V4"), "v4",                             np.where(mt.str.contains("V5"), "v5", "other"))    else:        df["model_family"] = "unknown"    for col in [C["label1"], C["label2"]]:        df[f"{col}_mature"] = df[col].notna().astype(int)    df["label_any_mature"] = (        df[f"{C['label1']}_mature"] | df[f"{C['label2']}_mature"]    ).astype(int)    df["default"] = np.where(        df["label_any_mature"] == 1,        np.fmax(df[C["label1"]].fillna(0), df[C["label2"]].fillna(0)),        np.nan,    )    return dfdf_2w = basic_clean(raw)print(f"After cleaning: {len(df_2w):,} rows, {df_2w[C['user_id']].nunique():,} users")# ── Load XGB models and score ─────────────────────────────────────────────feat_df_v4 = pd.read_excel(CFG["feature_path_v4"])feat_df_v5 = pd.read_excel(CFG["feature_path_v5"])def get_feat_list(df):    return df["索引"].dropna().tolist() if "索引" in df.columns else df.iloc[:, 0].dropna().tolist()CFG["feat_v4"]    = get_feat_list(feat_df_v4)CFG["feat_v5"]    = get_feat_list(feat_df_v5)CFG["feat_inter"] = sorted(set(CFG["feat_v4"]) & set(CFG["feat_v5"]))CFG["feat_union"] = sorted(set(CFG["feat_v4"]) | set(CFG["feat_v5"]))print(f"Features - V4:{len(CFG['feat_v4'])}, V5:{len(CFG['feat_v5'])}, "      f"inter:{len(CFG['feat_inter'])}, union:{len(CFG['feat_union'])}")bv4 = xgb.Booster(); bv4.load_model(CFG["model_path_v4"])bv5 = xgb.Booster(); bv5.load_model(CFG["model_path_v5"])def score_with_model(df, booster, flist, out_col):    X = pd.DataFrame(        {f: pd.to_numeric(df[f], errors="coerce").values if f in df.columns else np.nan         for f in flist}, index=df.index,    )    df[out_col] = booster.predict(xgb.DMatrix(X.values, feature_names=flist, missing=np.nan))    return dfdf_2w = score_with_model(df_2w, bv4, CFG["feat_v4"], "score_v4")df_2w = score_with_model(df_2w, bv5, CFG["feat_v5"], "score_v5")print("Scores computed.")# ── Overlap window & continuing users ─────────────────────────────────────dm = df_2w.groupby(["event_date", "model_family"]).size().unstack(fill_value=0)ov = dm.index[(dm.get("v4", pd.Series(dtype=int)) > 0) &              (dm.get("v5", pd.Series(dtype=int)) > 0)]if len(ov):    CFG["overlap_start"] = max(CFG["overlap_start"], ov.min())    CFG["overlap_end"]   = min(CFG["overlap_end"],   ov.max())df_sorted = df_2w.sort_values([C["user_id"], "event_date", C["event_date"]])users_pre  = set(df_sorted[df_sorted["event_date"] < CFG["overlap_start"]][C["user_id"]])users_in   = set(df_sorted[(df_sorted["event_date"] >= CFG["overlap_start"]) &                            (df_sorted["event_date"] <= CFG["overlap_end"])][C["user_id"]])continuing = users_pre & users_indf_aw = df_sorted[    (df_sorted["event_date"] >= CFG["overlap_start"]) &    (df_sorted["event_date"] <= CFG["overlap_end"]) &    (df_sorted[C["user_id"]].isin(continuing))].copy()# Anchor: first event in overlap window per continuing useranchor = df_aw.loc[df_aw.groupby(C["user_id"])["event_date"].idxmin()].copy()anchor = anchor.rename(columns={"event_date": "anchor_date"})df_all = df_sorted[df_sorted[C["user_id"]].isin(continuing)].copy()# Pre-anchor historypre_events = df_all.merge(anchor[[C["user_id"], "anchor_date"]], on=C["user_id"], how="inner")pre_events = pre_events[pre_events["event_date"] < pre_events["anchor_date"]].copy()def hist_summary(grp):    n_ev = len(grp)    return pd.Series(dict(        pre_n_events=n_ev,        pre_n_inc=grp["is_inc"].sum(),        pre_n_dec=grp["is_dec"].sum(),        pre_n_same=(grp["treat_dir"] == "same").sum(),        pre_cum_dr=grp["delta_rel"].sum(),        pre_last_dr=grp["delta_rel"].iloc[-1] if n_ev > 0 else 0.0,    ))hist_feat = (    pre_events.sort_values([C["user_id"], "event_date"])    .groupby(C["user_id"], group_keys=False)    .apply(hist_summary, include_groups=False)    .reset_index())# ── User-level survival table ─────────────────────────────────────────────first_def = (    df_all[df_all["default"] > 0].sort_values("event_date")    .groupby(C["user_id"])["event_date"].first().reset_index()    .rename(columns={"event_date": "default_date"}))last_obs = (    df_all.groupby(C["user_id"])["event_date"].max().reset_index()    .rename(columns={"event_date": "last_date"}))union_in_anchor = [f for f in CFG["feat_union"] if f in anchor.columns]us = (    anchor[[C["user_id"], "anchor_date", "model_family", "treat_dir",            C["bf_credit"], C["af_credit"], "score_v4", "score_v5",            "is_inc", "is_dec", "delta_abs", "delta_rel", "delta_log"] + union_in_anchor]    .merge(first_def, on=C["user_id"], how="left")    .merge(last_obs,  on=C["user_id"], how="left")    .merge(hist_feat, on=C["user_id"], how="left"))us = us[~(us["default_date"].notna() & (us["default_date"] < us["anchor_date"]))].copy()us["defaulted"] = (us["default_date"].notna() & (us["default_date"] >= us["anchor_date"])).astype(int)us["T"] = np.where(    us["defaulted"] == 1,    (us["default_date"] - us["anchor_date"]).dt.days,    (us["last_date"] - us["anchor_date"]).dt.days,).clip(1, CFG["max_followup_days"])us.loc[(us["defaulted"] == 1) &       ((us["default_date"] - us["anchor_date"]).dt.days > CFG["max_followup_days"]),       "defaulted"] = 0us["is_v5"]     = (us["model_family"] == "v5").astype(int)us["lc"]        = np.log1p(us[C["bf_credit"]])us["lc_z"]      = standardise(us["lc"])# Standardised treatment variables for user-levelus["delta_abs_z"]  = standardise(us["delta_abs"])us["delta_rel_z"]  = standardise(us["delta_rel"])# Decomposed: positive part (inc), negative part (dec)us["delta_pos_z"]  = standardise(us["delta_abs"].clip(lower=0))us["delta_neg_z"]  = standardise(us["delta_abs"].clip(upper=0))# Active scoreus["active_score"]     = np.where(us["is_v5"]==1, us["score_v5"], us["score_v4"])us["active_score_pct"] = us.groupby("is_v5")["active_score"].rank(pct=True)print(f"\nOverlap: {CFG['overlap_start'].date()} -> {CFG['overlap_end'].date()}")print(f"Continuing users: {len(continuing):,}")print(f"User-level table: {len(us):,}  V4={(us['is_v5']==0).sum()}  V5={(us['is_v5']==1).sum()}")print(f"Defaults: {us['defaulted'].sum()} ({us['defaulted'].mean():.1%})")print(f"Treatment: inc={(us['treat_dir']=='inc').sum()}, "      f"same={(us['treat_dir']=='same').sum()}, dec={(us['treat_dir']=='dec').sum()}")# ╔══════════════════════════════════════════════════════════════════════════╗# ║  SECTION C: KM CURVES & LOG-RANK                                          ║# ╚══════════════════════════════════════════════════════════════════════════╝fig, axes = plt.subplots(1, 3, figsize=(18, 5))for mf, c in [("v4", "C0"), ("v5", "C1")]:    s = us[us["model_family"] == mf]    km = kaplan_meier(s["T"], s["defaulted"])    axes[0].step(km["time"], km["survival"], where="post", lw=2, color=c,                 label=f"{mf} (n={len(s)}, DR={s['defaulted'].mean():.1%})")axes[0].set(xlabel="Days from anchor", ylabel="S(t)", title="KM: V4 vs V5"); axes[0].legend()for td, c in [("inc", "C1"), ("same", "C2"), ("dec", "C3")]:    s = us[us["treat_dir"] == td]    km = kaplan_meier(s["T"], s["defaulted"])    axes[1].step(km["time"], km["survival"], where="post", lw=2, color=c,                 label=f"{td} (n={len(s)}, DR={s['defaulted'].mean():.1%})")axes[1].set(xlabel="Days from anchor", ylabel="S(t)", title="KM: By treatment"); axes[1].legend()for mf, ls in [("v4", "-"), ("v5", "--")]:    for td, c in [("inc", "C1"), ("same", "C2"), ("dec", "C3")]:        s = us[(us["model_family"] == mf) & (us["treat_dir"] == td)]        if len(s) < 10: continue        km = kaplan_meier(s["T"], s["defaulted"])        axes[2].step(km["time"], km["survival"], where="post", lw=1.5,                     color=c, ls=ls, label=f"{mf}-{td} ({s['defaulted'].mean():.0%})")axes[2].set(xlabel="Days from anchor", ylabel="S(t)", title="KM: Model x Treatment")axes[2].legend(fontsize=6, ncol=2)fig.tight_layout(); plt.show()s4 = us[us["is_v5"] == 0]; s5 = us[us["is_v5"] == 1]chi2_lr, p_lr = logrank_test(s4["T"], s4["defaulted"], s5["T"], s5["defaulted"])print(f"Log-rank V4 vs V5: chi2={chi2_lr:.3f}, p={p_lr:.4f} {stars(p_lr)}")# ╔══════════════════════════════════════════════════════════════════════════╗# ║  SECTION D: m(X) CONSTRUCTION                                             ║# ╚══════════════════════════════════════════════════════════════════════════╝print("\n" + "="*78)print("  SECTION D: m(X) CONSTRUCTION")print("="*78)feat_mX_us = [f for f in CFG["feat_union"] if f in us.columns]print(f"  Feature set: feat_union, {len(feat_mX_us)} columns in user table")risk_oof_us, risk_auc_us, risk_std_us, n_feat_us = xgb_crossfit(    us, feat_mX_us, "defaulted", CFG,    group_col=C["user_id"], mcw=10, num_rounds=300,)us["mX"] = risk_oof_usus["mX"] = us["mX"].fillna(us["mX"].median())print(f"  m(X) OOF AUC = {risk_auc_us:.4f} +/- {risk_std_us:.4f}  (n_features={n_feat_us})")# ╔══════════════════════════════════════════════════════════════════════════╗# ║  SECTION E: COUNTING PROCESS + EXPOSURE CONSTRUCTION                      ║# ╚══════════════════════════════════════════════════════════════════════════╝print("\n" + "="*78)print("  SECTION E: COUNTING PROCESS + EXPOSURE CONSTRUCTION")print("="*78)anchor_info = anchor[[C["user_id"], "anchor_date", "model_family",                       C["bf_credit"], "score_v4", "score_v5"]].copy()ep = df_all.merge(anchor_info, on=C["user_id"], how="inner", suffixes=("", "_anc"))ep["day"] = (ep["event_date"] - ep["anchor_date"]).dt.daysep = ep[(ep["day"] >= 0) & (ep["day"] <= CFG["max_followup_days"])].copy()ep["event_default"] = (ep["default"] > 0).astype(int)ep = ep.sort_values([C["user_id"], "day", C["event_date"]])EPS = 0.01cp_rows = []for uid, grp in ep.groupby(C["user_id"]):    grp = grp.sort_values(["day", C["event_date"]]).reset_index(drop=True)    n = len(grp)    if n == 0: continue    # Effective times (handle same-day ties)    eff = np.empty(n, dtype=float)    for idx in range(n):        d = grp.iloc[idx]["day"]        K = (grp["day"] == d).sum()        k = int((grp["day"] == d).iloc[:idx+1].sum()) - 1        eff[idx] = d + (k + 1) / (K + 1)    for idx in range(n):        eff[idx] = max(eff[idx], EPS) if idx == 0 else max(eff[idx], eff[idx-1] + EPS)    base_credit = float(grp.iloc[0][C["bf_credit"]])    base_safe = max(base_credit, 1.0)    cum_pos_rel = cum_neg_rel = cum_net_rel = 0.0    dose_pos_rel = dose_neg_rel = dose_net_rel = 0.0    current_level = base_credit    prev_time = 0.0    for i in range(n):        row = grp.iloc[i]        current_time = eff[i]        duration = current_time - prev_time        excess_rel = (current_level - base_credit) / base_safe        # Dose: lagged (before this event)        dose_pos_rel_lag = dose_pos_rel        dose_neg_rel_lag = dose_neg_rel        dose_net_rel_lag = dose_net_rel        dose_pos_rel += max(0.0, excess_rel) * duration        dose_neg_rel += min(0.0, excess_rel) * duration  # negative values        dose_net_rel += excess_rel * duration        # Cumulative: lagged        cum_pos_rel_lag = cum_pos_rel        cum_neg_rel_lag = cum_neg_rel        cum_net_rel_lag = cum_net_rel        delta = float(row[C["af_credit"]]) - float(row[C["bf_credit"]])        delta_rel = delta / base_safe        cum_pos_rel += max(0.0, delta_rel)        cum_neg_rel += min(0.0, delta_rel)        cum_net_rel += delta_rel        current_level = float(row[C["af_credit"]])        prev_time = current_time        start = eff[i-1] if i > 0 else 0.0        ev = int(row["event_default"] == 1)        cp_rows.append({            "user_id": uid, "interval_idx": i,            "start": start, "stop": current_time, "event": ev,            "day": int(row["day"]),            "is_v5": int(str(row.get("model_family_anc", row.get("model_family", "v4"))) == "v5"),            "tv_is_inc": int(row["is_inc"]),            "tv_is_dec": int(row["is_dec"]),            "tv_delta_rel": delta_rel,            "tv_score_v4": float(row["score_v4"]),            "tv_score_v5": float(row["score_v5"]),            "base_credit": base_credit,            # Cumulative lag1            "cum_pos_rel_lag1": cum_pos_rel_lag,            "cum_neg_rel_lag1": cum_neg_rel_lag,            "cum_net_rel_lag1": cum_net_rel_lag,            # Dose lag1            "dose_pos_rel_lag1": dose_pos_rel_lag,            "dose_neg_rel_lag1": dose_neg_rel_lag,            "dose_net_rel_lag1": dose_net_rel_lag,        })        if ev == 1:            breakcp = pd.DataFrame(cp_rows)cp["t_mid"] = (cp["start"] + cp["stop"]) / 2cp["t_mid_sq"] = cp["t_mid"] ** 2# Lag1 scoresfor col in ["tv_score_v4", "tv_score_v5", "tv_is_inc", "tv_is_dec"]:    cp[f"{col}_lag1"] = cp.groupby("user_id")[col].shift(1)cp["tv_own_score"] = np.where(cp["is_v5"] == 1, cp["tv_score_v5"], cp["tv_score_v4"])cp["tv_own_score_lag1"] = cp.groupby("user_id")["tv_own_score"].shift(1)# Standardise exposure variables in cpfor col in ["cum_pos_rel_lag1", "cum_neg_rel_lag1", "cum_net_rel_lag1",            "dose_pos_rel_lag1", "dose_neg_rel_lag1", "dose_net_rel_lag1",            "tv_delta_rel"]:    cp[f"{col}_z"] = standardise(cp[col].fillna(0))# Standardise base creditcp["base_credit_z"] = standardise(np.log1p(cp["base_credit"]))# Merge features for m(X) at event levelfeat_in_ep = [f for f in feat_mX_us if f in ep.columns]ep_feat_df = ep[["user_id"] + feat_in_ep].copy()ep_feat_df["_rank"] = ep_feat_df.groupby("user_id").cumcount()cp["_rank"] = cp.groupby("user_id").cumcount()cp = cp.merge(ep_feat_df, on=["user_id", "_rank"], how="left", suffixes=("", "_ep"))feat_in_cp = [f for f in feat_in_ep if f in cp.columns]# m(X) at event levelrisk_oof_cp, risk_auc_cp, risk_std_cp, n_feat_cp = xgb_crossfit(    cp, feat_in_cp, "event", CFG,    group_col="user_id", mcw=30, num_rounds=300,)cp["mX"] = risk_oof_cp# Fill NaNfor col in ["tv_score_v4", "tv_score_v5", "tv_own_score", "mX",            "tv_score_v4_lag1", "tv_score_v5_lag1", "tv_own_score_lag1"]:    if col in cp.columns:        cp[col] = cp[col].fillna(cp[col].median())for col in ["cum_pos_rel_lag1", "cum_neg_rel_lag1", "cum_net_rel_lag1",            "dose_pos_rel_lag1", "dose_neg_rel_lag1", "dose_net_rel_lag1",            "cum_pos_rel_lag1_z", "cum_neg_rel_lag1_z", "cum_net_rel_lag1_z",            "dose_pos_rel_lag1_z", "dose_neg_rel_lag1_z", "dose_net_rel_lag1_z",            "tv_delta_rel_z", "base_credit_z"]:    if col in cp.columns:        cp[col] = cp[col].fillna(0.0)print(f"\nCounting process: {len(cp):,} intervals, "      f"{cp['user_id'].nunique():,} users, {cp['event'].sum()} defaults")print(f"m(X) event-level OOF AUC: {risk_auc_cp:.4f} +/- {risk_std_cp:.4f}")print(f"Stop <= Start violations: {(cp['stop'] <= cp['start']).sum()}")# ── Exposure distributions ────────────────────────────────────────────────fig, axes = plt.subplots(2, 3, figsize=(16, 8))for ax, col in zip(axes[0], ["cum_pos_rel_lag1", "cum_neg_rel_lag1", "cum_net_rel_lag1"]):    ax.hist(cp[col].dropna(), bins=50, alpha=.7); ax.set(title=col)for ax, col in zip(axes[1], ["dose_pos_rel_lag1", "dose_neg_rel_lag1", "dose_net_rel_lag1"]):    ax.hist(cp[col].dropna(), bins=50, alpha=.7); ax.set(title=col)fig.suptitle("Lag-1 Cumulative & Dose Exposure Distributions", fontsize=12)fig.tight_layout(); plt.show()# ╔══════════════════════════════════════════════════════════════════════════╗# ║  PART C: SCORE-TO-TREATMENT RULE ANALYSIS                                 ║# ║  C1: Absolute score mapping                                               ║# ║  C2: Percentile mapping                                                   ║# ║  C3: V5 calibration shift decomposition                                   ║# ╚══════════════════════════════════════════════════════════════════════════╝print("\n" + "="*78)print("  PART C: SCORE-TO-TREATMENT RULE ANALYSIS")print("="*78)print("  NOTE: Only overlap-window data used for rule learning.")# ── C1: Absolute score -> treatment ───────────────────────────────────────print("\n  -- C1: Absolute score -> treatment mapping --")fig, axes = plt.subplots(1, 2, figsize=(16, 5))for ax, score_col, score_label in [    (axes[0], "score_v4", "V4 score"),    (axes[1], "score_v5", "V5 score"),]:    for regime, c, lab in [(0, "C0", "V4 regime"), (1, "C1", "V5 regime")]:        sub = us[us["is_v5"] == regime].copy()        sub["_sbin"] = pd.qcut(sub[score_col], q=20, duplicates="drop")        grp = sub.groupby("_sbin", observed=True).agg(            inc_rate=("is_inc", "mean"),            dec_rate=("is_dec", "mean"),            n=("user_id", "count"),            score_mid=(score_col, "median"),        ).reset_index()        ax.plot(grp["score_mid"], grp["inc_rate"], "o-", color=c, label=f"{lab} P(inc)", ms=3)        ax.plot(grp["score_mid"], grp["dec_rate"], "s--", color=c, alpha=0.5,                label=f"{lab} P(dec)", ms=3)    ax.set(xlabel=score_label, ylabel="Treatment probability",           title=f"C1: Treatment mapping by {score_label}")    ax.legend(fontsize=7)fig.tight_layout(); plt.show()# Formal test: absolute score mappingfor score_col in ["score_v4", "score_v5"]:    m = safe_glm(f"is_inc ~ {score_col} * is_v5", us)    if m:        stata_table(m, f"C1: is_inc ~ {score_col} * is_v5  (absolute score mapping test)")# ── C2: Percentile mapping ───────────────────────────────────────────────print("\n  -- C2: Percentile -> treatment mapping --")# Compute within-regime percentileus["score_v4_pct"] = us.groupby("is_v5")["score_v4"].rank(pct=True)us["score_v5_pct"] = us.groupby("is_v5")["score_v5"].rank(pct=True)m_pct_base = safe_glm("is_inc ~ active_score_pct + is_v5", us)m_pct_full = safe_glm("is_inc ~ active_score_pct * is_v5", us)if m_pct_base:    stata_table(m_pct_base, "C2 [base]: is_inc ~ percentile + is_v5  (level-shift)")if m_pct_full:    stata_table(m_pct_full, "C2 [full]: is_inc ~ percentile * is_v5  (slope-diff)")# Percentile inc rate plotus["active_pct_d10"] = safe_qcut_rank(us["active_score_pct"], q=10, labels=False)pct_map = us.groupby(["active_pct_d10", "is_v5"]).agg(    inc_rate=("is_inc", "mean"), n=("user_id", "count")).reset_index()fig, axes = plt.subplots(1, 2, figsize=(14, 5))for regime, c, lab in [(0, "C0", "V4 regime"), (1, "C1", "V5 regime")]:    sub = pct_map[pct_map["is_v5"] == regime]    axes[0].plot(sub["active_pct_d10"], sub["inc_rate"], "o-", color=c, label=lab)axes[0].set(xlabel="Active score percentile decile", ylabel="Inc rate",            title="C2: Inc rate by percentile (within regime)")axes[0].legend()p4 = pct_map[pct_map["is_v5"]==0].set_index("active_pct_d10")["inc_rate"]p5 = pct_map[pct_map["is_v5"]==1].set_index("active_pct_d10")["inc_rate"]delta_pct = (p5 - p4).dropna()axes[1].bar(delta_pct.index, delta_pct.values*100, alpha=.7, color="C3")axes[1].axhline(0, c="k", lw=.8)axes[1].set(xlabel="Percentile decile", ylabel="V5-V4 inc rate (pp)",            title="C2: Inc rate lift V5 vs V4 at same percentile")fig.tight_layout(); plt.show()# ── C3: V5 calibration shift decomposition ────────────────────────────────print("\n  -- C3: V5 calibration shift -> excess inc decomposition --")# Score distribution comparisonfig, axes = plt.subplots(1, 2, figsize=(14, 5))for ax, score_col in [(axes[0], "score_v4"), (axes[1], "score_v5")]:    s_v4 = us.loc[us["is_v5"]==0, score_col].dropna()    s_v5 = us.loc[us["is_v5"]==1, score_col].dropna()    ks_stat, ks_p = ks_2samp(s_v4, s_v5)    ax.hist(s_v4, bins=40, alpha=.5, density=True, label=f"V4 regime (n={len(s_v4)})")    ax.hist(s_v5, bins=40, alpha=.5, density=True, label=f"V5 regime (n={len(s_v5)})")    ax.set(title=f"{score_col} by regime (KS={ks_stat:.3f}, p={ks_p:.4f} {stars(ks_p)})")    ax.legend(fontsize=7)    print(f"  KS [{score_col}]: stat={ks_stat:.4f}  p={ks_p:.4f} {stars(ks_p)}")fig.suptitle("C3: Score Distributions by Regime", fontsize=12)fig.tight_layout(); plt.show()# Counterfactual: for V5 users, what would V4 model assign?# (V4 score is already computed for all users)# Learn V4 rule: P(inc | v4_score) from V4-regime users onlyv4_users = us[us["is_v5"]==0].copy()v4_users["_sbin20"] = pd.qcut(v4_users["score_v4"], q=20, duplicates="drop")v4_rule = v4_users.groupby("_sbin20", observed=True).agg(    inc_rate=("is_inc", "mean"), score_lo=("score_v4", "min"),    score_hi=("score_v4", "max")).reset_index()# For V5 users, map their V4 score to V4 ruledef cf_inc_rate_v4rule(score_v4, rule_df):    """Counterfactual inc rate under V4 rule given V4 score."""    for _, row in rule_df.iterrows():        if row["score_lo"] <= score_v4 <= row["score_hi"]:            return row["inc_rate"]    return rule_df["inc_rate"].mean()v5_users = us[us["is_v5"]==1].copy()v5_users["cf_inc_rate_v4rule"] = v5_users["score_v4"].apply(    lambda s: cf_inc_rate_v4rule(s, v4_rule))actual_v5_inc_rate = v5_users["is_inc"].mean()cf_v4rule_inc_rate = v5_users["cf_inc_rate_v4rule"].mean()actual_v4_inc_rate = v4_users["is_inc"].mean()print(f"\n  Decomposition of excess inc for V5 users:")print(f"    Actual V4 inc rate        : {actual_v4_inc_rate:.4f}")print(f"    CF V4-rule on V5 users    : {cf_v4rule_inc_rate:.4f}")print(f"    Actual V5 inc rate        : {actual_v5_inc_rate:.4f}")print(f"    Total excess = {actual_v5_inc_rate - actual_v4_inc_rate:+.4f}")print(f"    Calibration channel       = {cf_v4rule_inc_rate - actual_v4_inc_rate:+.4f}")print(f"    Rule channel              = {actual_v5_inc_rate - cf_v4rule_inc_rate:+.4f}")# ╔══════════════════════════════════════════════════════════════════════════╗# ║  PART A: PERFORMATIVE PREDICTION EXISTENCE                                 ║# ╚══════════════════════════════════════════════════════════════════════════╝# ── A1: Event-level DML (User-level data, each user = one event) ──────────# NOTE: "Event-level" here means treating each user as one event/observation# NOT person-time hazard analysis (that's A2)print("\n" + "="*78)print("  A1: EVENT-LEVEL DML — Y ~ T + m(X)")print("="*78)print("  Goal: show performative prediction exists.")print("  Event-level: each user is one event/observation.")print("  Treatment T decomposed: max(D,0) [inc effect], min(D,0) [dec effect].")# --- A1a-d: User-level logistic models ---a1_specs = [    ("A1a", "defaulted ~ delta_abs_z + mX + lc_z",     "net delta (standardised)"),    ("A1b", "defaulted ~ delta_pos_z + delta_neg_z + mX + lc_z",     "decomposed: max(D,0) + min(D,0)"),    ("A1c", "defaulted ~ delta_pos_z + delta_neg_z + mX + lc_z + is_v5",     "decomposed + regime"),    ("A1d", "defaulted ~ delta_pos_z + delta_neg_z + mX + lc_z + score_v4 + score_v5",     "decomposed + both scores"),]for spec_id, fml, label in a1_specs:    m = safe_glm(fml, us, cluster_col=C["user_id"])    if m:        stata_table(m, f"[{spec_id}] {label}")# --- A1e: DML partialling-out ---print("\n  -- A1e: DML partialling-out (event-level: user as unit) --")Y_res = us["defaulted"].values - us["mX"].valuesfor treat_col, treat_label in [("delta_abs_z", "net delta"),                                 ("delta_pos_z", "max(D,0)"),                                 ("delta_neg_z", "min(D,0)")]:    oof_t, _, _, _ = xgb_crossfit(        us, feat_mX_us, treat_col, CFG,        group_col=C["user_id"], objective="reg:squarederror",        is_binary=False, mcw=10, num_rounds=200,    )    T_res = us[treat_col].values - oof_t    mask = np.isfinite(Y_res) & np.isfinite(T_res)    X_dml = T_res[mask].reshape(-1, 1)    y_dml = Y_res[mask]    lr = LinearRegression().fit(X_dml, y_dml)    theta = lr.coef_[0]    # Bootstrap SE (clustered by user — each user is one obs here)    rng = np.random.default_rng(CFG["seed"])    n_boot = 500    boot_thetas = []    for _ in range(n_boot):        idx = rng.choice(len(y_dml), len(y_dml), replace=True)        boot_thetas.append(LinearRegression().fit(X_dml[idx], y_dml[idx]).coef_[0])    se = np.std(boot_thetas)    z_val = theta / se if se > 0 else 0    p_val = 2 * (1 - stats.norm.cdf(abs(z_val)))    ci_lo = theta - 1.96 * se; ci_hi = theta + 1.96 * se    dml_table(f"A1e DML [{treat_label}]", theta, se, z_val, p_val, ci_lo, ci_hi,              n=int(mask.sum()))# ── A2: Event-level Hazard DML (person-time data) ─────────────────────────# NOTE: This is true person-time hazard analysisprint("\n" + "="*78)print("  A2: EVENT-LEVEL HAZARD DML — discrete-time hazard ~ T + m(X)")print("="*78)print("  Robustness check: performative prediction in hazard framework.")print("  T decomposed: event-level delta, inc/dec indicators.")cp_h = cp.dropna(subset=["mX"]).copy()a2_specs = [    ("A2a", "event ~ t_mid + t_mid_sq + mX + base_credit_z + tv_delta_rel_z",     "time + m(X) + credit + delta"),    ("A2b", "event ~ t_mid + t_mid_sq + mX + base_credit_z + tv_is_inc + tv_is_dec",     "time + m(X) + credit + inc/dec"),    ("A2c", "event ~ t_mid + t_mid_sq + mX + base_credit_z + tv_is_inc + tv_is_dec + is_v5",     "time + m(X) + credit + inc/dec + regime"),    ("A2d", "event ~ t_mid + t_mid_sq + mX + base_credit_z + tv_delta_rel_z + is_v5",     "time + m(X) + credit + delta + regime"),]for spec_id, fml, label in a2_specs:    m = safe_glm(fml, cp_h, cluster_col="user_id")    if m:        stata_table(m, f"[{spec_id}] {label}")# ── A3: Cumulative Exposure MSM ────────────────────────────────────────────print("\n" + "="*78)print("  A3: CUMULATIVE EXPOSURE MSM — True Time-Varying Treatment History")print("="*78)print("  Exposure: Cumulative credit changes up to time t")print("  MSM: Marginal Structural Model with IPTW for treatment history")# --- A3a-d: Conditional models with cumulative exposure ---a3_specs = [    ("A3a", "event ~ t_mid + t_mid_sq + mX + base_credit_z + cum_pos_rel_lag1_z + cum_neg_rel_lag1_z",     "conditional: cumulative pos/neg"),    ("A3b", "event ~ t_mid + t_mid_sq + mX + base_credit_z + cum_pos_rel_lag1_z + cum_neg_rel_lag1_z + is_v5",     "conditional: cumulative pos/neg + regime"),    ("A3c", "event ~ t_mid + t_mid_sq + mX + base_credit_z + cum_net_rel_lag1_z",     "conditional: cumulative net"),    ("A3d", "event ~ t_mid + t_mid_sq + mX + base_credit_z + cum_net_rel_lag1_z + is_v5",     "conditional: cumulative net + regime"),]for spec_id, fml, label in a3_specs:    m = safe_glm(fml, cp_h, cluster_col="user_id")    if m:        stata_table(m, f"[{spec_id}] {label}")# --- A3e: IPTW for cumulative exposure MSM ---print("\n  -- A3e: IPTW Marginal Structural Model --")print("  Treatment model: P(A_t | X_bar_t, A_bar_{t-1})")print("  Including treatment history in denominator model")# Treatment categorizationcp_h["tv_treat_cat"] = np.select(    [cp_h["tv_is_inc"] == 1, cp_h["tv_is_dec"] == 1],    [2, 0], default=1  # 0=dec, 1=same, 2=inc)# Denominator model: P(A_t | X_bar_t, A_bar_{t-1})# INCLUDES treatment history via cum_pos_rel_lag1_z, cum_neg_rel_lag1_ziptw_feats = feat_in_cp + ["t_mid", "is_v5", "base_credit_z",                            "cum_pos_rel_lag1_z", "cum_neg_rel_lag1_z"]iptw_feats = [f for f in iptw_feats if f in cp_h.columns]iptw_feats = list(dict.fromkeys(iptw_feats))print(f"  Building treatment propensity models with {len(iptw_feats)} features")print(f"  Including treatment history: cum_pos_rel_lag1_z, cum_neg_rel_lag1_z")# Build treatment propensity modelsfor cat_val, cat_label in [(2, "inc"), (0, "dec")]:    cp_h[f"_treat_is_{cat_label}"] = (cp_h["tv_treat_cat"] == cat_val).astype(int)    oof_denom, _, _, _ = xgb_crossfit(        cp_h, iptw_feats, f"_treat_is_{cat_label}", CFG,        group_col="user_id", mcw=30, num_rounds=200,    )    cp_h[f"_p_denom_{cat_label}"] = np.clip(oof_denom, 0.01, 0.99)# Numerator: P(A_t | A_bar_{t-1}) - marginal probabilitiesmarginal_inc = cp_h["tv_is_inc"].mean()marginal_dec = cp_h["tv_is_dec"].mean()marginal_same = 1 - marginal_inc - marginal_dec# Compute stabilised weights: SW_t = [P(A_t | A_bar_{t-1})] / [P(A_t | X_bar_t, A_bar_{t-1})]def compute_sw(row):    if row["tv_treat_cat"] == 2:        num, den = marginal_inc, row["_p_denom_inc"]    elif row["tv_treat_cat"] == 0:        num, den = marginal_dec, row["_p_denom_dec"]    else:        num = marginal_same        den = 1 - row["_p_denom_inc"] - row["_p_denom_dec"]    den = max(den, 0.01)    return num / dencp_h["_sw"] = cp_h.apply(compute_sw, axis=1)# Cumulative stabilized weights: CSW_t = ∏_{k=1}^{t} SW_k# This accounts for the full treatment historycp_h["_csw"] = cp_h.groupby("user_id")["_sw"].cumprod()# Truncate extreme weightsp1, p99 = cp_h["_csw"].quantile(0.01), cp_h["_csw"].quantile(0.99)cp_h["_csw_trunc"] = cp_h["_csw"].clip(p1, p99)print(f"  IPTW weight stats:")print(f"    mean={cp_h['_csw_trunc'].mean():.3f}, median={cp_h['_csw_trunc'].median():.3f}")print(f"    p5={cp_h['_csw_trunc'].quantile(0.05):.3f}, p95={cp_h['_csw_trunc'].quantile(0.95):.3f}")print(f"  Weights account for time-varying confounding and treatment history")# Weighted MSM: estimates marginal effects of cumulative exposuretry:    msm_fml = "event ~ t_mid + t_mid_sq + cum_pos_rel_lag1_z + cum_neg_rel_lag1_z"    m_msm = smf.glm(msm_fml, data=cp_h,                     family=sm.families.Binomial(),                     freq_weights=cp_h["_csw_trunc"].values).fit(cov_type="HC1")    stata_table(m_msm, "[A3e] IPTW MSM: cumulative pos/neg")    msm_fml2 = "event ~ t_mid + t_mid_sq + cum_net_rel_lag1_z"    m_msm2 = smf.glm(msm_fml2, data=cp_h,                      family=sm.families.Binomial(),                      freq_weights=cp_h["_csw_trunc"].values).fit(cov_type="HC1")    stata_table(m_msm2, "[A3f] IPTW MSM: cumulative net")    print("  ✓ MSM successfully estimated marginal effects of cumulative exposure")except Exception as e:    print(f"  ✗ IPTW MSM failed: {e}")# ── A4: Cumulative Dose MSM ────────────────────────────────────────────────print("\n" + "="*78)print("  A4: CUMULATIVE DOSE MSM — Time-Integrated Exposure")print("="*78)print("  Dose: Time-integrated exposure (area under curve of credit changes)")a4_specs = [    ("A4a", "event ~ t_mid + t_mid_sq + mX + base_credit_z + dose_pos_rel_lag1_z + dose_neg_rel_lag1_z",     "conditional: dose pos/neg"),    ("A4b", "event ~ t_mid + t_mid_sq + mX + base_credit_z + dose_pos_rel_lag1_z + dose_neg_rel_lag1_z + is_v5",     "conditional: dose + regime"),    ("A4c", "event ~ t_mid + t_mid_sq + mX + base_credit_z + dose_net_rel_lag1_z",     "conditional: dose net"),    ("A4d", "event ~ t_mid + t_mid_sq + mX + base_credit_z + dose_net_rel_lag1_z + is_v5",     "conditional: dose net + regime"),]for spec_id, fml, label in a4_specs:    m = safe_glm(fml, cp_h, cluster_col="user_id")    if m:        stata_table(m, f"[{spec_id}] {label}")# IPTW MSM for dose (reuses weights from A3)try:    msm_fml3 = "event ~ t_mid + t_mid_sq + dose_pos_rel_lag1_z + dose_neg_rel_lag1_z"    m_msm3 = smf.glm(msm_fml3, data=cp_h,                      family=sm.families.Binomial(),                      freq_weights=cp_h["_csw_trunc"].values).fit(cov_type="HC1")    stata_table(m_msm3, "[A4e] IPTW MSM: dose pos/neg")    msm_fml4 = "event ~ t_mid + t_mid_sq + dose_net_rel_lag1_z"    m_msm4 = smf.glm(msm_fml4, data=cp_h,                      family=sm.families.Binomial(),                      freq_weights=cp_h["_csw_trunc"].values).fit(cov_type="HC1")    stata_table(m_msm4, "[A4f] IPTW MSM: dose net")    print("  ✓ MSM successfully estimated marginal effects of cumulative dose")except Exception as e:    print(f"  ✗ IPTW MSM for dose failed: {e}")# ╔══════════════════════════════════════════════════════════════════════════╗# ║  PART B: EXCESS-INC LOCAL CAUSAL EFFECT                                    ║# ╚══════════════════════════════════════════════════════════════════════════╝# ── B1: Estimate individual-level excess inc probability ──────────────────print("\n" + "="*78)print("  B1: EXCESS-INC PROPENSITY ESTIMATION")print("="*78)print("  p_excess_i = P(inc|X, regime=V5) - P(inc|X, regime=V4)")# Strategy: train inc propensity model with is_v5 as feature,# then predict with is_v5=0 and is_v5=1inc_feats_base = [f for f in feat_mX_us if f in us.columns]inc_feats_all = inc_feats_base + ["is_v5", "score_v4", "score_v5", "lc_z",                                   "pre_n_inc", "pre_n_dec", "pre_cum_dr"]inc_feats_all = [f for f in inc_feats_all if f in us.columns]oof_inc, auc_inc, std_inc, n_inc_feat = xgb_crossfit(    us, inc_feats_all, "is_inc", CFG,    group_col=C["user_id"], mcw=10, num_rounds=300,)us["p_inc_oof"] = oof_incprint(f"  Inc propensity OOF AUC: {auc_inc:.4f} +/- {std_inc:.4f}  (n_features={n_inc_feat})")# Counterfactual predictionscf_feats = [f for f in inc_feats_all if f in us.columns]cf_X = us[cf_feats].values.astype(np.float32)cf_y = us["is_inc"].values.astype(int)xgb_params_cf = dict(    objective="binary:logistic", eval_metric="logloss",    max_depth=4, learning_rate=0.05, subsample=0.8,    colsample_bytree=0.6, min_child_weight=10,    reg_alpha=0.1, reg_lambda=1.0,    device=CFG.get("xgb_device", "cpu"), verbosity=0,)dt_full = xgb.DMatrix(cf_X, label=cf_y, feature_names=cf_feats, missing=np.nan)bst_cf = xgb.train(xgb_params_cf, dt_full, num_boost_round=300, verbose_eval=False)is_v5_idx = cf_feats.index("is_v5")X_v4 = cf_X.copy(); X_v4[:, is_v5_idx] = 0X_v5 = cf_X.copy(); X_v5[:, is_v5_idx] = 1us["p_inc_if_v4"] = bst_cf.predict(xgb.DMatrix(X_v4, feature_names=cf_feats, missing=np.nan))us["p_inc_if_v5"] = bst_cf.predict(xgb.DMatrix(X_v5, feature_names=cf_feats, missing=np.nan))us["delta_p_inc"] = us["p_inc_if_v5"] - us["p_inc_if_v4"]us["delta_p_inc_z"] = standardise(us["delta_p_inc"])print(f"  Mean p_inc_if_v4  : {us['p_inc_if_v4'].mean():.4f}")print(f"  Mean p_inc_if_v5  : {us['p_inc_if_v5'].mean():.4f}")print(f"  Mean delta_p_inc  : {us['delta_p_inc'].mean():+.4f}")print(f"  Pr(delta_p > 0)   : {(us['delta_p_inc'] > 0).mean():.3f}")fig, axes = plt.subplots(1, 3, figsize=(16, 4))axes[0].hist(us["delta_p_inc"], bins=50, alpha=.8)axes[0].axvline(0, c="k", lw=.8); axes[0].axvline(us["delta_p_inc"].mean(), c="r", lw=1.5)axes[0].set(title="B1: delta_p_inc distribution", xlabel="P(inc|V5) - P(inc|V4)")axes[1].scatter(us["p_inc_if_v4"], us["p_inc_if_v5"],                alpha=.1, s=4, c=us["is_v5"], cmap="coolwarm")axes[1].plot([0,1],[0,1],"k--",alpha=.3)axes[1].set(xlabel="P(inc|V4)", ylabel="P(inc|V5)", title="B1: Counterfactual propensities")axes[2].scatter(us["score_v4"], us["delta_p_inc"], alpha=.15, s=4, c=us["is_v5"], cmap="coolwarm")axes[2].axhline(0, c="k", lw=.8)axes[2].set(xlabel="V4 score", ylabel="delta_p_inc", title="B1: delta_p vs V4 score")fig.tight_layout(); plt.show()# ── B2: Matching with 4 Feature Sets ───────────────────────────────────────print("\n" + "="*78)print("  B2: MATCHING — EXCESS-INC USERS VS CONTROL")print("="*78)print("  Testing 4 different feature sets:")print("    1. V4 features only (baseline)")print("    2. V5 features only")print("    3. V4 AND V5 features (both scores)")print("    4. V4 OR V5 features (union of all features)")print("="*78)# Define the 4 feature setsfeature_set_configs = [    ("V4_only", ["score_v4", "lc_z"], "V4 score + baseline characteristics"),    ("V5_only", ["score_v5", "lc_z"], "V5 score + baseline characteristics"),    ("V4_and_V5", ["score_v4", "score_v5", "lc_z"], "Both V4 and V5 scores + baseline"),    ("V4_or_V5", ["score_v4", "score_v5", "lc_z", "pre_cum_dr", "pre_n_inc", "pre_n_dec"],     "Union: both scores + all baseline features"),]# Run matching for each threshold and feature set combinationfor tau in [0.05, 0.10, 0.20]:    print(f"\n{'='*78}")    print(f"  THRESHOLD tau = {tau:.2f}")    print(f"{'='*78}")    # Treatment: V5 inc users with high excess probability    treated = us[(us["is_v5"] == 1) & (us["is_inc"] == 1) &                 (us["delta_p_inc"] > tau)].copy()    # Control: V4 non-inc users (potential counterfactuals)    control = us[(us["is_v5"] == 0) & (us["is_inc"] == 0)].copy()    print(f"  Treated (V5 inc, delta_p>{tau}): {len(treated)}")    print(f"  Control (V4 non-inc): {len(control)}")    if len(treated) < 10 or len(control) < 10:        print("  Too few units for matching. Skipping all feature sets.")        continue    # Try each feature set    for feat_set_name, base_match_vars, feat_set_desc in feature_set_configs:        print(f"\n  --- Feature Set: {feat_set_name} ---")        print(f"  Description: {feat_set_desc}")        # Filter to available features        match_vars = [v for v in base_match_vars if v in treated.columns and v in control.columns]        if len(match_vars) == 0:            print(f"  No matching variables available. Skipping.")            continue        print(f"  Matching on: {', '.join(match_vars)}")        # Propensity score matching        match_df = pd.concat([            treated[match_vars + ["defaulted", "T"]].assign(_treated=1),            control[match_vars + ["defaulted", "T"]].assign(_treated=0),        ]).dropna(subset=match_vars)        if len(match_df) < 20:            print(f"  Insufficient data after dropna. Skipping.")            continue        try:            ps_model = LogisticRegression(max_iter=1000, C=1.0)            ps_model.fit(match_df[match_vars], match_df["_treated"])            match_df["_ps"] = ps_model.predict_proba(match_df[match_vars])[:, 1]        except Exception as e:            print(f"  PS model failed: {e}")            continue        # Check overlap        ps_t = match_df.loc[match_df["_treated"]==1, "_ps"]        ps_c = match_df.loc[match_df["_treated"]==0, "_ps"]        print(f"  PS range treated: [{ps_t.min():.3f}, {ps_t.max():.3f}]")        print(f"  PS range control: [{ps_c.min():.3f}, {ps_c.max():.3f}]")        # 1:1 nearest-neighbour matching with caliper        caliper = 0.2 * match_df["_ps"].std()        t_idx = match_df[match_df["_treated"]==1].index.values        c_idx = match_df[match_df["_treated"]==0].index.values        t_ps = match_df.loc[t_idx, "_ps"].values.reshape(-1, 1)        c_ps = match_df.loc[c_idx, "_ps"].values.reshape(-1, 1)        nn = NearestNeighbors(n_neighbors=1, metric="euclidean")        nn.fit(c_ps)        dists, indices = nn.kneighbors(t_ps)        matched_pairs = []        used_controls = set()        for i, (d, j) in enumerate(zip(dists.ravel(), indices.ravel())):            if d <= caliper and j not in used_controls:                matched_pairs.append((t_idx[i], c_idx[j]))                used_controls.add(j)        if len(matched_pairs) < 10:            print(f"  Only {len(matched_pairs)} matched pairs. Skipping.")            continue        t_matched = match_df.loc[[p[0] for p in matched_pairs]]        c_matched = match_df.loc[[p[1] for p in matched_pairs]]        print(f"  Matched pairs: {len(matched_pairs)}")        # Balance check        print(f"  Balance (SMD):")        for v in match_vars + ["_ps"]:            smd = compute_smd(t_matched[v], c_matched[v])            status = '[OK]' if abs(smd) < 0.1 else '[IMBALANCED]'            print(f"    {v}: SMD = {smd:.4f} {status}")        # Outcome comparison - LATE estimate        dr_t = t_matched["defaulted"].mean()        dr_c = c_matched["defaulted"].mean()        rd = dr_t - dr_c        se_rd = np.sqrt(dr_t*(1-dr_t)/len(t_matched) + dr_c*(1-dr_c)/len(c_matched))        z_rd = rd / se_rd if se_rd > 0 else 0        p_rd = 2 * (1 - stats.norm.cdf(abs(z_rd)))        print(f"\n  Default rate treated: {dr_t:.4f}")        print(f"  Default rate control: {dr_c:.4f}")        dml_table(f"B2 [{feat_set_name}] (tau={tau:.2f}): LATE estimate",                  rd, se_rd, z_rd, p_rd,                  rd - 1.96*se_rd, rd + 1.96*se_rd,                  n=len(matched_pairs),                  extra_info={"Treated": len(t_matched), "Control": len(c_matched),                              "DR_treated": f"{dr_t:.4f}", "DR_control": f"{dr_c:.4f}",                              "Features": feat_set_name})# ── B3: Hazard with excess probability ────────────────────────────────────print("\n" + "="*78)print("  B3: HAZARD WITH EXCESS PROBABILITY")print("="*78)# B3 implementation would go here...print("\n" + "="*78)print("  CORRECTIONS SUMMARY (V3 - FINAL):")print("="*78)print("  1. A1: Event-level DML with user-level data (each user = one event) ✓")print("  2. A2: Person-time hazard DML (robustness check) ✓")print("  3. B2: Tests 4 feature sets (V4, V5, V4&V5, V4|V5) ✓")print("  4. A3/A4: MSM with IPTW including treatment history ✓")print("     - Denominator: P(A_t | X_bar_t, A_bar_{t-1})")print("     - Numerator: P(A_t | A_bar_{t-1})")print("     - Cumulative weights: CSW_t = ∏_{k=1}^{t} SW_k")print("     - Marginal structural parameters estimated via weighted GLM")print("="*78)# ── B3: Hazard with excess probability ────────────────────────────────────print("\n" + "="*78)print("  B3: HAZARD WITH EXCESS PROBABILITY")print("="*78)# Merge delta_p into cpcp_le = cp_h.merge(us[["user_id", "delta_p_inc", "delta_p_inc_z"]],                    on="user_id", how="left")cp_le["delta_p_inc_z"] = cp_le["delta_p_inc_z"].fillna(0.0)b3_specs = [    ("B3a", "event ~ t_mid + t_mid_sq + is_v5 + mX + base_credit_z + delta_p_inc_z",     "regime + m(X) + credit + delta_p"),    ("B3b", ("event ~ t_mid + t_mid_sq + is_v5 + mX + base_credit_z + tv_is_inc "             "+ delta_p_inc_z + tv_is_inc:delta_p_inc_z"),     "regime + m(X) + credit + inc + delta_p + inc*delta_p"),    ("B3c", ("event ~ t_mid + t_mid_sq + is_v5 + mX + base_credit_z + tv_is_inc + tv_is_dec "             "+ delta_p_inc_z + tv_is_inc:delta_p_inc_z + tv_is_dec:delta_p_inc_z"),     "regime + m(X) + credit + inc + dec + delta_p + interactions"),    ("B3d", ("event ~ t_mid + t_mid_sq + is_v5 + mX + base_credit_z "             "+ cum_pos_rel_lag1_z + cum_neg_rel_lag1_z + delta_p_inc_z"),     "regime + m(X) + credit + cum + delta_p"),]for spec_id, fml, label in b3_specs:    m = safe_glm(fml, cp_le, cluster_col="user_id")    if m:        stata_table(m, f"[{spec_id}] {label}")# ╔══════════════════════════════════════════════════════════════════════════╗# ║  SUMMARY TABLE — all key coefficients across models                       ║# ╚══════════════════════════════════════════════════════════════════════════╝print("\n" + "="*78)print("  FINAL SUMMARY")print("="*78)print("""Key findings summary:  Part C: Score-to-treatment rule analysis    - C1: Tests whether absolute score -> treatment mapping differs V4 vs V5    - C2: Tests whether percentile -> treatment mapping differs V4 vs V5    - C3: Decomposes excess inc into calibration vs rule channels  Part A: Performative prediction existence    - A1: Event-level DML shows whether T has residual association with Y beyond m(X)    - A2: Hazard-level DML confirms in discrete-time framework    - A3: Cumulative delta credit (conditional + IPTW) tests user-path exposure    - A4: Cumulative dose (conditional + IPTW) tests time-integrated exposure  Part B: Excess-inc local causal effect    - B1: Identifies users pushed into inc by V5 regime (delta_p_inc)    - B2: Matches excess-inc users to V4 non-inc controls -> LATE estimate    - B3: Hazard models with delta_p_inc as heterogeneity variable  All credit variables standardised. Results in STATA format (* p<0.10).  Default proxy: event_date as default date.  Sample: continuing users in overlap window.""")print("="*78)print("  DONE — Export this notebook as HTML for full results.")print("="*78)